In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE = '/content/drive/MyDrive/HiARN/data'
os.makedirs(DRIVE, exist_ok=True)

BRISC_CSV      = f'{DRIVE}/brisc_features.csv'

BRISC_DIR  = '/content/brisc2025'

print(f'Drive         : {DRIVE}')
print(f'BRISC CSV     : {BRISC_CSV}')

In [3]:
import subprocess, sys, time

def _install(label, *args):
    t0 = time.time()
    print(f'  {label}...', end=' ', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + list(args),
                       capture_output=True, text=True)
    ok = r.returncode == 0
    print(f'OK [{time.time()-t0:.0f}s]' if ok else f'FAILED\n{r.stderr[-200:]}')

print('=== Пакети ===')
_install('pyradiomics', 'git+https://github.com/AIM-Harvard/pyradiomics.git')
_install('SimpleITK',             'SimpleITK')
_install('opencv-python-headless','opencv-python-headless')
_install('tqdm',                  'tqdm')
print('Done')

=== Пакети ===
  pyradiomics... OK [35s]
  SimpleITK... OK [3s]
  opencv-python-headless... OK [3s]
  tqdm... OK [4s]
Done


In [4]:
import os, shutil
from google.colab import files as colab_files

KAGGLE_DIR        = os.path.expanduser('~/.kaggle')
KAGGLE_JSON_DRIVE = f'{DRIVE}/kaggle.json'
KAGGLE_JSON_LOCAL = f'{KAGGLE_DIR}/kaggle.json'
os.makedirs(KAGGLE_DIR, exist_ok=True)

kaggle_ready = False

if os.path.exists(KAGGLE_JSON_DRIVE):
    shutil.copy(KAGGLE_JSON_DRIVE, KAGGLE_JSON_LOCAL)
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
    print('kaggle.json: знайдено на Drive ✓')
    kaggle_ready = True
elif os.path.exists(KAGGLE_JSON_LOCAL):
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
    print('kaggle.json: знайдено локально ✓')
    kaggle_ready = True
else:
    print('kaggle.json не знайдено.')
    print('Отримайте на: https://www.kaggle.com/settings → API → "Create New Token"')
    uploaded = colab_files.upload()
    if 'kaggle.json' in uploaded:
        with open(KAGGLE_JSON_LOCAL, 'wb') as f:
            f.write(uploaded['kaggle.json'])
        os.chmod(KAGGLE_JSON_LOCAL, 0o600)
        try:
            shutil.copy(KAGGLE_JSON_LOCAL, KAGGLE_JSON_DRIVE)
            print(f'Копія збережена на Drive: {KAGGLE_JSON_DRIVE}')
        except Exception:
            pass
        print('kaggle.json: завантажено ✓')
        kaggle_ready = True
    else:
        print('❌ kaggle.json не обрано — перезапустіть Cell 3')

assert kaggle_ready, 'Kaggle credentials не налаштовано'

kaggle.json: знайдено на Drive ✓


In [ ]:
import subprocess, time
from pathlib import Path

def kaggle_dl(slug, dest, marker):
    if Path(marker).exists():
        print(f'  SKIP (вже є): {marker}')
        return True
    print(f'  {slug} → {dest} ...', flush=True)
    t0 = time.time()
    r  = subprocess.run(['kaggle', 'datasets', 'download', '-d', slug,
                         '-p', dest, '--unzip'])
    if r.returncode == 0:
        print(f'  ✓ [{time.time()-t0:.0f}s]')
        return True
    print(f'  ❌ exit={r.returncode}')
    return False

print('=== Завантаження датасетів ===')

kaggle_dl('briscdataset/brisc2025',
          BRISC_DIR,
          '/content/brisc2025/brisc2025/segmentation_task')


print('\n=== Перевірка ===')
for name, path, pat in [
    ('BRISC',  '/content/brisc2025', '**/*.jpg'),
]:
    n = len(list(Path(path).rglob(pat))) if Path(path).exists() else 0
    print(f'  {name:8s}: {n} файлів')

In [ ]:
import time, logging, warnings, cv2
import numpy as np
import SimpleITK as sitk
from tqdm.notebook import tqdm
from radiomics import featureextractor
import pandas as pd

logging.getLogger('radiomics').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

PARAMS = {
    'imageType': {'Original': {}},
    'featureClass': {'firstorder': [], 'glcm': [], 'glszm': [], 'shape2D': []},
    'setting': {
        'binWidth': 25, 'normalize': True, 'normalizeScale': 100,
        'removeOutliers': 3.0, 'force2D': True, 'force2Ddimension': 0,
    },
}

extractor = featureextractor.RadiomicsFeatureExtractor(**PARAMS)
print('PyRadiomics ініціалізовано ✓')

CLASS_NAMES = {0: 'glioma', 1: 'meningioma', 2: 'pituitary', 3: 'no_tumor'}

def fmt_time(sec):
    m, s = divmod(int(sec), 60)
    h, m = divmod(m, 60)
    if h:  return f'{h}h{m:02d}m{s:02d}s'
    if m:  return f'{m}m{s:02d}s'
    return f'{s}s'

def pseudo_mask(img_arr):
    h, w = img_arr.shape[:2]
    msk  = np.zeros((h, w), dtype=np.int32)
    msk[int(h*.4):int(h*.6), int(w*.4):int(w*.6)] = 1
    return msk

def extract_features(img_arr, mask_arr):
    img_sitk  = sitk.GetImageFromArray(img_arr.astype(np.float32))
    mask_sitk = sitk.GetImageFromArray(mask_arr.astype(np.int32))
    result    = extractor.execute(img_sitk, mask_sitk)
    return {k: float(v) for k, v in result.items()
            if k.startswith('original_') and not isinstance(v, str)}

def save_csv(rows, csv_path, name):
    df        = pd.DataFrame(rows).fillna(0.0)
    feat_cols = [c for c in df.columns if c.startswith('original_')]
    print(f'\n{name}: {df.shape[0]} зразків × {len(feat_cols)} ознак')
    vc = df['label'].value_counts().sort_index()
    for lbl, cnt in vc.items():
        print(f'  клас {int(lbl)} ({CLASS_NAMES.get(int(lbl),"?")}): {cnt}')
    nan_n = df[feat_cols].isna().sum().sum()
    if nan_n:
        print(f'  NaN замінено нулями: {nan_n}')
    df.to_csv(csv_path, index=False)
    sz = os.path.getsize(csv_path) / 1024
    print(f'  Збережено: {csv_path}  ({sz:.0f} KB) ✓')
    return df

print('Спільні функції ініціалізовано ✓')
print('  pseudo_mask : центральний 40-60% (legacy, BRISC no_tumor fallback)')

In [8]:
import os, re, cv2
from pathlib import Path
from collections import Counter

if os.path.exists(BRISC_CSV):
    print(f'Кеш знайдено: {BRISC_CSV}  (видаліть файл для повторного обрахунку)')
else:
    _cands = ['/content/brisc2025/brisc2025',
              '/content/brisc2025',
              '/content/brisc2025/brisc2025/brisc2025']
    BRISC_ROOT = next((c for c in _cands if Path(c, 'segmentation_task').exists()), None)
    assert BRISC_ROOT, 'BRISC_ROOT не знайдено — запустіть Cell 4'
    print(f'BRISC_ROOT: {BRISC_ROOT}')

    CODE_MAP = {'gl': 0, 'me': 1, 'pi': 2}
    def _parse_code(name):
        m = re.search(r'_(gl|me|pi)_', name, re.IGNORECASE)
        if m:
            code = m.group(1).lower()
            return CODE_MAP.get(code)
        return None

    B = Path(BRISC_ROOT)
    tasks, n_skipped = [], 0
    for split in ('train', 'test'):
        img_dir  = B / 'segmentation_task' / split / 'images'
        mask_dir = B / 'segmentation_task' / split / 'masks'
        if not img_dir.exists():
            print(f'  SKIP: {img_dir}')
            continue
        for ip in sorted(img_dir.glob('*.jpg')):
            lbl = _parse_code(ip.name)
            if lbl is None:
                n_skipped += 1
                continue
            mp = mask_dir / (ip.stem + '.png')
            if mp.exists():
                tasks.append((str(ip), str(mp), lbl, ip.name))

    if n_skipped:
        print(f'  ⚠ Пропущено (невідомий формат імені): {n_skipped} файлів')

    NO_TUMOR_LIMIT, n_no_tumor = 1000, 0
    for split in ('train', 'test'):
        nt_dir = B / 'classification_task' / split / 'no_tumor'
        if nt_dir.exists():
            for ip in sorted(nt_dir.glob('*.jpg')):
                if n_no_tumor >= NO_TUMOR_LIMIT:
                    break
                tasks.append((str(ip), None, 3, ip.name))
                n_no_tumor += 1

    lbl_cnt = Counter(t[2] for t in tasks)
    print(f'Завдань: {len(tasks)}  розподіл: {dict(lbl_cnt)}')
    print('Орієнтовний час: ~30-90 хв')

    BRISC_CKPT = BRISC_CSV.replace('.csv', '_checkpoint.csv')
    rows, _done = [], set()
    if os.path.exists(BRISC_CKPT):
        _ckpt = pd.read_csv(BRISC_CKPT)
        rows = _ckpt.to_dict('records')
        _done = set(_ckpt['filename'].tolist())
        print(f'Checkpoint відновлено: {len(rows)} рядків ({len(_done)} файлів)')

    n_ok, n_err = len(rows), 0
    t0   = time.time()
    pbar = tqdm(tasks, unit='img')
    for img_path, mask_path, label, filename in pbar:
        if filename in _done:
            continue
        try:
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise ValueError('imread None')
            if mask_path is not None:
                msk_raw = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                _, msk  = cv2.threshold(msk_raw, 127, 1, cv2.THRESH_BINARY)
                if msk.sum() == 0:
                    msk = pseudo_mask(img).astype(np.uint8)
                msk = msk.astype(np.int32)
            else:
                msk = pseudo_mask(img)
            feats = extract_features(img, msk)
            feats['label']    = label
            feats['filename'] = filename
            rows.append(feats)
            n_ok += 1
            if n_ok % 500 == 0:
                pd.DataFrame(rows).fillna(0.0).to_csv(BRISC_CKPT, index=False)
                tqdm.write(f'[ckpt] {n_ok} рядків збережено')
        except Exception as e:
            n_err += 1
            if n_err <= 5 or n_err % 200 == 0:
                tqdm.write(f'[err {n_err}] {filename}: {str(e)[:80]}')
        pbar.set_postfix({'ok': n_ok, 'err': n_err}, refresh=False)
    pbar.close()
    print(f'Витягування: {n_ok} OK / {n_err} помилок  [{fmt_time(time.time()-t0)}]')

    save_csv(rows, BRISC_CSV, 'BRISC 2025')
    if os.path.exists(BRISC_CKPT):
        os.remove(BRISC_CKPT)
        print('Checkpoint видалено ✓')

print('\n>>> Cell 6 done')

BRISC_ROOT: /content/brisc2025/brisc2025
Завдань: 5793  розподіл: {0: 1401, 1: 1635, 2: 1757, 3: 1000}
Орієнтовний час: ~30-90 хв


  0%|          | 0/5793 [00:00<?, ?img/s]

[ckpt] 500 рядків збережено
[ckpt] 1000 рядків збережено
[ckpt] 1500 рядків збережено
[ckpt] 2000 рядків збережено
[ckpt] 2500 рядків збережено
[ckpt] 3000 рядків збережено
[ckpt] 3500 рядків збережено
[ckpt] 4000 рядків збережено
[ckpt] 4500 рядків збережено
[ckpt] 5000 рядків збережено
[ckpt] 5500 рядків збережено
Витягування: 5793 OK / 0 помилок  58s]

BRISC 2025: 5793 зразків × 93 ознак
  клас 0 (glioma): 1401
  клас 1 (meningioma): 1635
  клас 2 (pituitary): 1757
  клас 3 (no_tumor): 1000
  Збережено: /content/drive/MyDrive/HiARN/data/brisc_features.csv  (9501 KB) ✓
Checkpoint видалено ✓

>>> Cell 6 done
